# ProteomeLM: protein–protein interaction prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitbol-Lab/ProteomeLM/blob/main/notebooks/ppi_prediction_efficient.ipynb)

Predict protein–protein interactions (PPIs) across a whole proteome with [ProteomeLM](https://www.pnas.org/doi/10.1073/pnas.2524201123). Every protein is embedded with ESM-C, ProteomeLM reads the whole proteome at once (one token per protein), and each pair of proteins is scored from the attention between them.

**How to use**

1. **Setup**: run the first code cell. On Colab, first pick a GPU runtime (*Runtime → Change runtime type → T4 GPU*); the cell installs ProteomeLM (1–3 min). If Colab asks you to restart the session afterwards, do it and run the cells again.
2. **Settings**: choose the proteins (a STRING organism, your own FASTA, a UniProt taxon or a list of UniProt accessions) and what to score: the best partners of some query proteins, specific pairs, a random sample of pairs, or all pairs.
3. **Run**: *Runtime → Run all* (or run the remaining cells in order). Each step says what it did, and errors say which setting to change. The results are shown as tables and figures and saved as CSV, PNG and SVG files (downloaded automatically on Colab).

The default settings rank the partners of RNA polymerase β (*rpoB*) and FtsZ among the ~4,100 proteins of *E. coli* K-12 in a few minutes on a Colab T4. The first run also downloads the ESM-C 600M weights (2.3 GB) and ProteomeLM-S (74 MB).

**Proteome size and memory.** ProteomeLM looks at all protein pairs at once, so its memory grows with the square of the number of proteins *N*: about 4 bytes × 8 heads × *N*² for ProteomeLM-S, i.e. ~0.6 GB for a bacterium (4k proteins), ~1.3 GB for yeast (6k) and ~13 GB for human (20k, needs a GPU with ≥16 GB). The notebook checks this before the slow steps and falls back to the CPU when the GPU is too small.

In [ ]:
#@title 1. Setup: install (Colab), check the GPU, import { display-mode: "form" }
# Where Colab installs ProteomeLM from. The bundled scoring models (data/interactomes/)
# are also downloaded from here when the notebook does not run inside a clone of the repo.
PROTEOMELM_REPO_URL = "https://github.com/Bitbol-Lab/ProteomeLM"  #@param {type:"string"}
PROTEOMELM_REF = "main"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)  # before torch is imported
warnings.filterwarnings("ignore", message=".*IProgress not found.*")  # tqdm without ipywidgets
IN_COLAB = "google.colab" in sys.modules


def _find_local_clone():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "proteomelm" / "__init__.py").is_file() and (folder / "pyproject.toml").is_file():
            return folder
    return None


LOCAL_CLONE = _find_local_clone()
if LOCAL_CLONE is not None:
    sys.path.insert(0, str(LOCAL_CLONE))  # use the code of the clone this notebook lives in
elif importlib.util.find_spec("proteomelm") is None:
    package = f"proteomelm[colab] @ git+{PROTEOMELM_REPO_URL}@{PROTEOMELM_REF}"
    if not IN_COLAB:
        raise ImportError(f'ProteomeLM is not installed. Run: pip install "{package}"')
    print("Installing ProteomeLM and its dependencies (1-3 min)...")
    install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], capture_output=True, text=True)
    if install.returncode != 0:
        print(install.stdout[-3000:], install.stderr[-3000:])
        raise RuntimeError("pip install failed (see the log above).")
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from proteomelm.ppi.notebook_inference import (
    backbone_attention_shape, compute_candidate_pairs, iter_pair_chunks, load_proteomelm_backbone,
    load_supervised_ppi_model, plan_notebook_run, prepare_notebook_proteome, primary_score_column,
    resolve_device, resolve_pair_queries, resolve_protein_queries, score_pair_chunks, summarize_top_partners,
)
from proteomelm.ppi.notebook_runtime import (
    attach_sparse_string_scores, build_config_form, configure_notebook_logging, get_cache_dir,
    load_logistic_model, normalize_notebook_config, offer_result_downloads, prepare_notebook_runtime_data,
    resolve_model_file, save_notebook_results, scoring_model_warnings, upload_fasta_in_colab,
)
from proteomelm.ppi.notebook_plots import (
    plot_head_auroc, plot_partner_network, plot_score_comparison, plot_string_agreement, plot_top_partners,
    show_figure,
)

try:
    import ipywidgets  # noqa: F401  (optional: interactive settings form)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

configure_notebook_logging()
torch.set_grad_enabled(False)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

import proteomelm
print(f"ProteomeLM code: {Path(proteomelm.__file__).parent}")
print(f"Work directory (downloads, embeddings, results): {get_cache_dir()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory / 1e9:.0f} GB)")
else:
    print("No GPU found: embedding a whole proteome with ESM-C on CPU is slow (hours for thousands of proteins)."
          + (" On Colab: Runtime > Change runtime type > T4 GPU, then run all cells again." if IN_COLAB else ""))

## 2. Settings

Edit the values below (on Colab, use the form on the right). Only the fields that match your choices are used:

- **Proteins**: `STRING organism` downloads a proteome from [STRING](https://string-db.org) by taxon id; `Local FASTA` reads your file (on Colab, leave the path empty to upload one); `UniProt taxon` downloads the reviewed (Swiss-Prot) proteins of a taxon; `UniProt accessions` uses just the listed proteins. ProteomeLM scores a pair *in the context of all loaded proteins*, so whole proteomes work best.
- **What to score**: partners of `query_proteins` (gene names such as `rpoB`, or ids), the `explicit_pairs` you list, a random sample of pairs, or all pairs (N² / 2 pairs: large for big proteomes).
- **Models**: `logreg_model` turns attention into a PPI score (`human` and `pathogens` were fit on ProteomeLM-S attention; `none` ranks by raw attention). `supervised_model` adds a second score from a supervised network on attention and ProteomeLM protein embeddings, and the ranking then uses it: `multispecies` (recommended: trained on human, *E. coli* and yeast pairs), `dscript` (human pairs only) or `bernett` (a strictly sequence-dissimilar human split; transfers poorly to other organisms). On 18 bacteria absent from training, `multispecies` ranks 72% of the interactions found by an AlphaFold screen of these bacteria in the top 1% of all pairs, against about 40% for the attention scores. On *E. coli* and yeast themselves, `multispecies` has seen those species' interactions during training, so their scores there are optimistic. `compare_with_string` adds each pair's STRING score, for comparison.
- **Advanced**: devices (`auto` = GPU if available), pairs scored per ProteomeLM pass, CPU fallback, output folder.

Outside Colab, the cell also shows an interactive form (if `ipywidgets` is installed) that edits the same settings.

In [ ]:
#@title 2. Settings { display-mode: "form" }
#@markdown ### Proteins
data_source = "STRING organism"  #@param ["STRING organism", "Local FASTA", "UniProt taxon", "UniProt accessions"]
#@markdown STRING organism: NCBI taxon id as used by STRING (511145 = *E. coli* K-12, 559292 = *S. cerevisiae*, 9606 = human).
string_id = "511145"  #@param {type:"string"}
#@markdown Local FASTA: path to a protein FASTA file (on Colab, leave empty to upload one).
local_fasta_path = ""  #@param {type:"string"}
#@markdown UniProt taxon: NCBI taxon id; its reviewed (Swiss-Prot) proteins are used.
taxon_id = ""  #@param {type:"string"}
#@markdown UniProt accessions: comma-separated list (these proteins form the whole context).
uniprot_ids = ""  #@param {type:"string"}

#@markdown ### What to score
scoring_mode = "Query proteins vs proteome"  #@param ["Query proteins vs proteome", "Explicit protein pairs", "Random sample of pairs", "All pairs"]
#@markdown Query proteins: gene names or protein ids, comma-separated.
query_proteins = "rpoB, ftsZ"  #@param {type:"string"}
#@markdown Explicit pairs: `a,b` pairs separated by `;`, e.g. `rpoB,rpoC; ftsZ,ftsA`.
explicit_pairs = ""  #@param {type:"string"}
sampled_pairs = 100000  #@param {type:"integer"}
#@markdown Number of top predictions to show (per query protein).
top_k = 20  #@param {type:"integer"}

#@markdown ### Models
checkpoint = "Bitbol-Lab/ProteomeLM-S"  #@param ["Bitbol-Lab/ProteomeLM-S", "Bitbol-Lab/ProteomeLM-XS", "Bitbol-Lab/ProteomeLM-M", "Bitbol-Lab/ProteomeLM-L"] {allow-input: true}
logreg_model = "human"  #@param ["human", "pathogens", "none"] {allow-input: true}
supervised_model = "none"  #@param ["none", "multispecies (recommended)", "dscript", "bernett"]
compare_with_string = True  #@param {type:"boolean"}

#@markdown ### Advanced
esm_device = "auto"  #@param {type:"string"}
proteomelm_device = "auto"  #@param {type:"string"}
pair_chunk_size = 1000000  #@param {type:"integer"}
#@markdown Run ProteomeLM on CPU when the proteome does not fit in free GPU memory:
auto_adjust = True  #@param {type:"boolean"}
#@markdown Output folder (empty = `results/` in the work directory printed by the setup cell).
output_dir = ""  #@param {type:"string"}

CONFIG = {
    "data_source_mode": data_source,
    "string_id": string_id,
    "local_fasta_path": local_fasta_path,
    "taxon_id": taxon_id,
    "uniprot_ids_text": uniprot_ids,
    "query_mode": scoring_mode,
    "query_proteins_text": query_proteins,
    "explicit_pairs_text": explicit_pairs,
    "sampled_pairs": sampled_pairs,
    "top_k": top_k,
    "checkpoint": checkpoint,
    "logreg_model": logreg_model,
    "supervised_model": supervised_model,
    "compare_with_string": compare_with_string,
    "esm_device": esm_device,
    "proteomelm_device": proteomelm_device,
    "pair_chunk_size": pair_chunk_size,
    "auto_adjust": auto_adjust,
    "output_dir": output_dir,
}

# Optional interactive form (Jupyter/VS Code with ipywidgets). It edits CONFIG in place.
if HAS_WIDGETS and not IN_COLAB:
    display(build_config_form(CONFIG))

## 3. Run

### Load the proteins and check the plan
Downloads or reads the proteins, finds the query proteins, and checks that the run fits in memory before the slow steps. Downloads are cached in the work directory and reused on the next run.

In [ ]:
RUN = normalize_notebook_config(CONFIG, require_local_fasta=not IN_COLAB)
if RUN["data_source_mode"] == "local_path" and not RUN["local_fasta_path"]:  # Colab: ask for a file
    CONFIG["local_fasta_path"] = upload_fasta_in_colab()
    RUN = normalize_notebook_config(CONFIG)
DATA = prepare_notebook_runtime_data(RUN)
PROTEIN_IDS = DATA.record_ids
DISPLAY_LABELS = DATA.display_labels
MODE = RUN["query_mode"]
print(f"Loaded {len(PROTEIN_IDS):,} proteins of {DATA.organism_name} ({DATA.fasta_for_inference}).")

QUERY_INDICES, PAIR_INDICES = [], None
if MODE == "query_proteins":
    QUERY_INDICES = resolve_protein_queries(PROTEIN_IDS, RUN["query_proteins"], aliases=DISPLAY_LABELS)
    print("Query proteins:", "; ".join(f"{PROTEIN_IDS[i]} ({DISPLAY_LABELS.get(PROTEIN_IDS[i], '')})" for i in QUERY_INDICES))
elif MODE == "explicit_pairs":
    PAIR_INDICES = resolve_pair_queries(PROTEIN_IDS, RUN["explicit_pairs"], aliases=DISPLAY_LABELS)
    print(f"{len(PAIR_INDICES)} protein pairs to score.")

# Scoring models: fetched now so a missing or incompatible file fails before the slow steps.
N_LAYERS, N_HEADS = backbone_attention_shape(RUN["checkpoint"], RUN["base_model_path"] or None)
LOGREG_MODEL = load_logistic_model(resolve_model_file(RUN["logreg_model"], "logreg", PROTEOMELM_REPO_URL, PROTEOMELM_REF))
for message in scoring_model_warnings(RUN["checkpoint"], N_LAYERS * N_HEADS, LOGREG_MODEL, RUN["logreg_model"], RUN["supervised_model"]):
    print("Note:", message)

N_PAIRS = compute_candidate_pairs(
    mode=MODE, n_proteins=len(PROTEIN_IDS), query_indices=QUERY_INDICES,
    explicit_pair_labels=RUN["explicit_pairs"], sampled_pairs=RUN["sampled_pairs"],
)
ESM_DEVICE = resolve_device(RUN["esm_device"])
PLM_DEVICE, PLAN, PLAN_NOTES = plan_notebook_run(
    n_proteins=len(PROTEIN_IDS), mode=MODE, candidate_pairs=N_PAIRS, n_layers=N_LAYERS, n_heads=N_HEADS,
    proteomelm_device=RUN["proteomelm_device"], chunk_size=RUN["pair_chunk_size"],
    include_string_annotations=RUN["compare_with_string"],
    include_supervised_ppi=RUN["supervised_model"] != "none", auto_adjust=RUN["auto_adjust"],
)
print()
print(PLAN.summary())
for note in PLAN_NOTES:
    print("Note:", note)
if ESM_DEVICE == "cpu" and len(PROTEIN_IDS) > 500:
    print(f"Note: ESM-C will embed {len(PROTEIN_IDS):,} proteins on CPU; this can take hours. A GPU is strongly recommended.")
if not PLAN.fits:
    raise RuntimeError("This run does not fit in memory; see the PROBLEM lines above and change the settings.")

### Embed the proteome and run ProteomeLM
ESM-C 600M embeds every protein (the slow step, cached for the next run), then ProteomeLM reads the whole proteome.

In [ ]:
start = time.time()
try:
    BACKBONE = load_proteomelm_backbone(RUN["checkpoint"], base_model_path=RUN["base_model_path"] or None, device=PLM_DEVICE)
    PROTEOME = prepare_notebook_proteome(
        # LoRA adapters: protein embeddings come from the base model, attention from the adapter.
        checkpoint=RUN["base_model_path"] or RUN["checkpoint"],
        fasta_file=DATA.fasta_for_inference,
        encoded_genome_file=DATA.encoded_genome_file,
        esm_device=ESM_DEVICE,
        proteomelm_device=PLM_DEVICE,
        reload_if_possible=True,
        model=None if RUN["base_model_path"] else BACKBONE,  # reuse the loaded backbone (not a LoRA-wrapped one)
    )
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError(
        "Out of GPU memory. Set proteomelm_device='cpu' (and/or esm_device='cpu'), use a smaller proteome, "
        "or a runtime with a larger GPU."
    ) from exc
if PROTEOME.labels != PROTEIN_IDS:
    raise RuntimeError("The embedded proteome does not match the loaded proteins; delete the embeddings/ folder in the work directory and re-run.")

SUPERVISED_MODEL = None
if RUN["supervised_model"] != "none":
    SUPERVISED_MODEL = load_supervised_ppi_model(
        resolve_model_file(RUN["supervised_model"], "supervised", PROTEOMELM_REPO_URL, PROTEOMELM_REF),
        protein_embed_dim=PROTEOME.protein_embeddings.shape[1],
        pair_feature_dim=N_LAYERS * N_HEADS,
        device=PLM_DEVICE,
        expected_backbone=RUN["checkpoint"],
    )
print(f"Proteome ready: {PROTEOME.n_proteins:,} proteins, ProteomeLM on {PLM_DEVICE} ({time.time() - start:.0f} s).")

### Score protein pairs
One ProteomeLM pass per chunk of pairs extracts the attention between the two proteins of each pair (all layers and heads), which the scoring models turn into interaction scores.

In [ ]:
start = time.time()
PAIR_CHUNKS = iter_pair_chunks(
    PROTEOME.n_proteins,
    chunk_size=RUN["pair_chunk_size"],
    explicit_pairs=PAIR_INDICES,
    query_indices=QUERY_INDICES if MODE == "query_proteins" else None,
    sampled_pairs=RUN["sampled_pairs"] if MODE == "sampled_all_pairs" else None,
    show_progress=True,
    progress_desc="Scoring pairs",
)
RESULTS = score_pair_chunks(
    BACKBONE, PROTEOME, PAIR_CHUNKS, device=PLM_DEVICE,
    logistic_model=LOGREG_MODEL, supervised_model=SUPERVISED_MODEL, display_labels=DISPLAY_LABELS,
)
SCORE_COLUMN = primary_score_column(RESULTS)
print(f"Scored {len(RESULTS):,} pairs in {time.time() - start:.0f} s (ranked by {SCORE_COLUMN}).")

### Compare with STRING (optional)
Adds `string_score`, the STRING combined score of each pair (0–1000; 0 = no STRING link). Only available for STRING and UniProt-taxon inputs (UniProt ids are mapped to STRING with BLAST+, which must be installed).

In [ ]:
if not RUN["compare_with_string"]:
    print("Skipped (compare_with_string is off).")
elif DATA.annotation_organism_id is None:
    print("Skipped: the STRING comparison needs a STRING organism or UniProt taxon as protein source.")
else:
    RESULTS = attach_sparse_string_scores(RESULTS, DATA.fasta_for_inference, DATA.annotation_organism_id, DATA.mapping_required)
    if "string_score" in RESULTS.columns:
        top = RESULTS.head(max(RUN["top_k"], 1))
        print(f"Pairs with a STRING score >= 400: {(top['string_score'] >= 400).mean():.0%} of the top {len(top)} "
              f"predictions vs {(RESULTS['string_score'] >= 400).mean():.1%} of all {len(RESULTS):,} scored pairs.")

### Results

In [ ]:
SCORE_COLUMNS = [c for c in ("unsupervised_score", "supervised_score", "string_score") if c in RESULTS.columns]
if MODE == "query_proteins":
    TOP = summarize_top_partners(RESULTS, QUERY_INDICES, RUN["top_k"], score_column=SCORE_COLUMN)
    for query_name, table in TOP.groupby("query", sort=False):
        print(f"\nTop {len(table)} predicted partners of {query_name} (by {SCORE_COLUMN})")
        display(table.drop(columns=["query", "query_id"]).set_index("rank"))
else:
    shown = RESULTS if MODE == "explicit_pairs" else RESULTS.head(RUN["top_k"])
    TOP = shown[["protein_a_label", "protein_b_label", *SCORE_COLUMNS, "protein_a", "protein_b"]]
    print(f"Top {len(TOP)} of {len(RESULTS):,} scored pairs (by {SCORE_COLUMN})")
    display(TOP.head(500))

### Save

In [ ]:
SAVED = save_notebook_results(
    RESULTS, RUN, DATA.organism_name, top_df=TOP, labels=PROTEOME.labels, output_dir=RUN["output_dir"] or None,
)
for kind, path in SAVED.items():
    print(f"{kind:>9}: {path} ({path.stat().st_size / 1e6:.1f} MB)")
FIGURE_PREFIX = SAVED["all_pairs"].with_suffix("")  # figures are saved next to the results (PNG + SVG)
FIGURES = []

## 4. Figures

Each figure is shown here and saved next to the results as PNG and SVG. Figures that need STRING scores or a supervised model are skipped with a one-line note when those are missing; very large result tables are subsampled for plotting.

### Top predicted partners
One panel per query protein (or the best pairs overall in the other modes). The x axis is how far each partner's score stands out from all of that query's pairs, in standard deviations, so different score types compare. With STRING scores, dark blue marks high-confidence STRING links, light blue medium ones, and gray pairs STRING does not support: known complexes at the top are a sanity check, gray pairs at the top are candidate new interactions (or false positives).

In [ ]:
FIGURES += show_figure(plot_top_partners(RESULTS, QUERY_INDICES, top_k=min(RUN["top_k"], 30)), "top_partners", FIGURE_PREFIX)

### Agreement with STRING
Left: the fraction of the top-N predictions that STRING supports, against the same fraction among all scored pairs (thin lines, what a random ranking gets). A useful ranking starts far above the background and converges to it as N grows. Right: ROC curve against STRING ≥ 400 for each score; the diagonal is chance.

In [ ]:
FIGURES += show_figure(plot_string_agreement(RESULTS), "string_agreement", FIGURE_PREFIX)

### Which attention heads carry the interaction signal
AUROC of each attention head on its own (layers × heads) for separating STRING-supported pairs from the rest, on up to 100k scored pairs (one extra ProteomeLM pass). Blue heads attend more between interacting proteins, red heads less, gray is chance. The logistic-regression score combines all of them.

In [ ]:
FIGURES += show_figure(plot_head_auroc(RESULTS, BACKBONE, PROTEOME, device=PLM_DEVICE), "head_auroc", FIGURE_PREFIX)

### Partner network (query proteins)
Each query as a hub with its top partners around it, clockwise by rank; partners in the top list of several queries sit between them, which highlights shared neighbourhoods. Edge color shows STRING support.

In [ ]:
FIGURES += show_figure(plot_partner_network(RESULTS, QUERY_INDICES, top_k=min(RUN["top_k"], 20)), "network", FIGURE_PREFIX)

### Attention vs supervised scores (with a supervised model)
Percentile rank of each pair under both scores. Pairs in the top-right corner are ranked high by both; the Spearman correlation says how much the two rankings agree.

In [ ]:
FIGURES += show_figure(plot_score_comparison(RESULTS), "score_comparison", FIGURE_PREFIX)

### Download (Colab)
On Colab this downloads the CSV files and a zip of the figures to your computer; elsewhere it prints the folder they are in.

In [ ]:
offer_result_downloads(SAVED, FIGURES)

## Output columns

| Column | Meaning |
|---|---|
| `protein_a`, `protein_b` | protein ids (FASTA record ids, e.g. STRING `511145.b3987` or UniProt `sp\|P0A8V2\|RPOB_ECOLI`) |
| `protein_a_label`, `protein_b_label` | readable names (gene \| protein name) from STRING or UniProt when available |
| `unsupervised_score` | ProteomeLM attention score from the logistic regression (raw summed attention when `logreg_model = "none"`). Use it to rank pairs: attention weights are small, so the bundled models give values just above 0.5 and the absolute value is not a calibrated probability |
| `supervised_score` | score of the supervised model (only when `supervised_model` is set); the ranking uses it when present |
| `string_score` | STRING combined score, 0–1000 (0 = no STRING link); only with `compare_with_string` |

Files: `<name>.csv` holds every scored pair, `<name>_top.csv` the table shown above, `<name>.pkl` the results with the settings used, and `<name>_<figure>.png` / `.svg` the figures (zipped as `<name>_figures.zip` for download). Above 1 M pairs, `<name>.csv` keeps only protein ids and the names go to `<name>_proteins.csv`.

**Tips.** Re-running is fast: downloads and ESM-C embeddings are cached in the work directory (delete it to free space). To screen a large proteome, start with query proteins or a random sample of pairs; all pairs of a 4k-protein bacterium is ~8.6 M rows (~1 min on a GPU, ~4 GB of RAM, ~0.3 GB CSV). The bundled scoring models were fit on ProteomeLM-S attention; with other sizes, prefer `logreg_model = "none"` (raw attention ranking).

**Citation.** Malbranke, Zalaffi & Bitbol, *ProteomeLM: A proteome-scale language model enables accurate and rapid prediction of protein-protein interactions and gene essentiality across taxa*, PNAS 123 (21), e2524201123 (2026). [doi:10.1073/pnas.2524201123](https://www.pnas.org/doi/10.1073/pnas.2524201123)